[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/01_crawl_storage/01_crawl_storage.ipynb)

# 01. `crawl-storage-example` 동행 노트북

> 대상 프로젝트: [`example-projects/crawl-storage-example`](../../../example-projects/crawl-storage-example) (A파트-1: 수집/저장)
> · 다른 선택지: [ALTERNATIVES.md](../../../example-projects/crawl-storage-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

프로젝트 하나를 통째로 넘겨받았다고 해봅시다. 파일이 4개 있습니다.
어디부터 열어야 할까요? 이 함수는 왜 이렇게 생겼을까요? 데이터는 어디로 흘러갈까요?

**도구를 아는 것과 남의 프로젝트를 읽는 것은 다른 능력입니다.**
`requests`를 쓸 줄 알아도, 남이 짜둔 크롤러를 이어받아 고치는 건 또 다른 일입니다.

이 노트북은 `crawl-storage-example`을 **처음부터 끝까지 같이 읽습니다.**
설명을 옮겨 적지 않고, 진짜 프로젝트 파일을 열어서 보여주고 그 안의 함수를
직접 import해서 돌려봅니다. **PostgreSQL 없이도요.**

읽으면서 계속 던질 질문은 하나입니다. **"왜 이렇게 짰을까?"**

### 이 프로젝트가 하는 일

사내 규정 챗봇을 만들려면 먼저 규정 문서를 모아야 합니다. 이 프로젝트가 그 첫 단계입니다.

```
웹 URL 목록 -> 페이지 방문 -> 본문 추출 -> PostgreSQL에 원본 그대로 저장
```

여기서 자주 나오는 질문 하나. **왜 [크롤링](../../../glossary.md#crawling)하자마자 바로 검색엔진에 넣지 않을까요?**

원본을 따로 보관해두면, 나중에 "[청킹](../../../glossary.md#chunking) 크기를 500자에서 300자로 바꿔보자"고 할 때
사이트를 다시 긁을 필요가 없습니다. DB에 있는 원본으로 다시 처리하면 되니까요.
크롤링은 느리고, 상대 서버에 부담을 주고, 사이트가 사라지면 다시 못 가져옵니다.
**한 번 긁은 원본은 금이라고 생각하고 그대로 보관**하는 것이 이 계층이 존재하는 이유입니다.

정제·청킹·색인은 다음 프로젝트인 [`02_preprocess`](../02_preprocess/02_preprocess.ipynb)가 맡습니다.

```
[01 crawl-storage] → 02 preprocess → 03 document-input → 04 rag-regulation
       여기
```

이번 장에서 배우는 것

- 낯선 프로젝트를 **어떤 순서로 읽는지** (파일 목록 → 메인 함수 → 그 아래로)
- 원본을 그대로 보관하는 계층이 왜 따로 있는지
- `extract_text_from_html()`이 `<script>`를 지우는 이유와, 안 지우면 검색이 어떻게 망가지는지
- [UPSERT](../../../glossary.md#postgresql)(`ON CONFLICT DO UPDATE`)가 없으면 왜 폐지된 조항이 검색되는지
- 실패를 격리하는 `try/except`와 `time.sleep()`의 의미
- **몽키패칭으로 DB 없이 전체 흐름을 실행하는 법** (테스트에서 실제로 쓰는 기법)

**소요 시간**: 30~40분. **PostgreSQL도 API 키도 없이 끝까지 실행됩니다.**
DB는 SQLite로, 저장 함수는 몽키패칭으로 대체합니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 첫 셀이 저장소를 통째로 내려받습니다(Colab 기준 10~20초).
- 코드 셀 앞에는 **지금 무엇을 할 것인지**를 적어두었습니다. 셀 뒤에는 두 가지가 붙습니다 —
  `show()`로 프로젝트 소스를 펼친 뒤에는 **코드에서 짚을 곳**이, 실제로 돌려본 뒤에는
  **결과 읽는 법**이 나옵니다.
- 6번 실습에서 **실제 네트워크 요청**이 한 번 나갑니다(`example.com`). 오프라인이면 그 셀만 건너뛰세요.
- 파이썬 기본 문법(함수, 딕셔너리, `with`)은 안다고 가정합니다. 라이브러리 사용법은 가정하지 않습니다.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](../../../troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q requests beautifulsoup4 python-dotenv psycopg2-binary
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "crawl-storage-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

## 1. 먼저 전체 지도를 본다

낯선 프로젝트를 받으면 파일 목록부터 봅니다. 이 프로젝트는 파일이 4개뿐이라 지도가 단순합니다.

In [ ]:
for name in sorted(os.listdir(SRC)):
    path = os.path.join(SRC, name)
    lines = len(open(path, encoding="utf-8").read().splitlines())
    print(f"  {name:<15} {lines:>3}줄")

| 파일 | 역할 | 비유 |
|---|---|---|
| `targets.py` | 어디를 긁을지 URL 목록 | 장 볼 목록 |
| `config.py` | 설정값 (DB 주소, 딜레이) | 준비물 |
| `crawl.py` | 실제로 긁어오는 메인 | 장 보러 가는 사람 |
| `db.py` | DB에 넣는 방법 | 창고 정리 담당 |

**읽는 순서는 `crawl.py`의 `main()`부터입니다.** 메인 흐름을 먼저 훑고,
거기서 부르는 함수를 필요할 때 따라 들어가는 게 남의 코드를 읽는 가장 빠른 길입니다.
아래에서 그 순서대로 갑니다.

## 2. `crawl.py`의 `main()` — 전체 흐름

코드 읽기부터 하겠습니다. 진짜 파일입니다.

In [ ]:
show("crawl.py", grep="^def |^    for |^        crawl_one|^    init_db")

**코드에서 짚을 곳** — 줄거리가 다 나왔습니다.

```
init_db()                     테이블 없으면 만들고
for url in CRAWL_TARGETS:     목록을 하나씩 돌면서
    crawl_one(url)            긁어서 저장하고
    time.sleep(...)           잠깐 쉰다
```

여기서 눈여겨볼 곳이 두 군데 있습니다. 둘 다 **실패와 예의**에 관한 것입니다.

In [ ]:
show("crawl.py", grep="except|sleep|HEADERS =|raise_for_status")

**코드에서 짚을 곳**

**① `try/except`로 감싼 이유** — URL 100개를 긁는데 3번째가 404를 내면 어떻게 될까요?
감싸지 않으면 거기서 프로그램이 죽고, 나머지 97개는 시도조차 못 합니다.
그래서 한 건의 실패가 전체를 멈추지 않도록 잡아서 출력만 하고 다음으로 넘어갑니다.

**② `time.sleep()`을 넣은 이유** — 요청을 쉬지 않고 연속으로 쏘면 상대 서버에 부담이 되고,
심하면 IP가 차단됩니다. 크롤러를 만들 때의 기본 예의이자, 내 크롤러를 지키는 방법이기도 합니다.

`HEADERS`에 브라우저인 척하는 User-Agent를 붙이는 것, `raise_for_status()`로 실패를
즉시 예외로 바꾸는 것도 같은 맥락입니다. **성공 경로보다 실패 경로를 먼저 설계한 코드**입니다.

## 3. `config.py` — 설정을 코드에서 분리하기

이제 `main()`이 쓰는 값들이 어디서 오는지 봅니다.

In [ ]:
show("config.py")

**코드에서 짚을 곳**

`os.getenv("이름", "기본값")` 패턴이 핵심입니다. `.env` 파일에 값이 있으면 그걸 쓰고,
없으면 기본값으로 돌아갑니다. 덕분에 **코드를 안 고치고도 환경마다 다르게 동작**시킬 수 있습니다.

실제로 import해서 값을 확인해봅시다. `.env`가 없으니 기본값이 나올 겁니다.

In [ ]:
import config

print("DATABASE_URL       :", config.DATABASE_URL)
print("CRAWL_DELAY_SECONDS:", config.CRAWL_DELAY_SECONDS)

# 환경변수를 넣으면 값이 바뀌는지 확인해봅니다. (실제 .env 파일을 쓰는 것과 같은 효과)
os.environ["CRAWL_DELAY_SECONDS"] = "2.5"
import importlib
importlib.reload(config)
print("\n환경변수를 넣은 뒤 :", config.CRAWL_DELAY_SECONDS)

> 💡 `OPENAI_API_KEY`처럼 **없으면 안 되는 값**은 `os.environ["..."]`로 읽어서 즉시 에러를 냅니다.
> `DATABASE_URL`처럼 **기본값으로 돌아가도 되는 값**은 `os.getenv(...)`를 씁니다.
> 이 프로젝트에는 전자가 없지만, `preprocess-example`과 `rag-regulation-example`에는 있습니다.
> 두 방식을 구분해서 쓰는 게 포인트입니다.

## 4. `extract_text_from_html()` — 실제로 돌려보기

`crawl_one()`이 부르는 함수 중 가장 내용이 많은 부분입니다. 먼저 코드를 읽고,
**프로젝트 파일에서 직접 import해서** 돌려보겠습니다.

In [ ]:
show("crawl.py", grep="def extract_text_from_html", )
print("---")
show("crawl.py", 38, 57)

In [ ]:
# 진짜 프로젝트 코드를 그대로 import합니다. (설명용으로 베껴 쓴 게 아닙니다)
from crawl import extract_text_from_html

messy_html = """
<html>
  <head>
    <style>body { color: red; }</style>
    <script>console.log("이건 본문이 아닙니다");</script>
  </head>
  <body>
    <nav>홈 | 소개 | 규정</nav>
    <h1>제9조(근로시간)</h1>

    <p>1주간의 소정근로시간은 40시간으로 한다.</p>


    <p>1일의 소정근로시간은 8시간으로 한다.</p>
  </body>
</html>
"""

print(extract_text_from_html(messy_html))

**결과 읽는 법**

`<script>`와 `<style>` 내용이 사라졌고, 빈 줄이 정리됐습니다. 이 두 가지가 이 함수의 전부입니다.

**왜 script/style을 지울까요?** 지우지 않으면 자바스크립트 코드가 그대로 "본문"으로 저장되고,
나중에 검색했을 때 `console.log`가 규정 내용인 것처럼 튀어나옵니다.
**검색 품질은 여기서부터 시작**입니다. 쓰레기를 넣으면 쓰레기가 나옵니다.

`separator="\n"`을 준 이유도 직접 확인해볼 수 있습니다. 빼면 어떻게 되는지 보세요.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(messy_html, "html.parser")
for tag in soup(["script", "style"]):
    tag.decompose()

print("separator 없이:")
print(repr(soup.get_text()[:80]))
print("\nseparator='\\n' 사용:")
print(repr(soup.get_text(separator="\n")[:80]))

**결과 읽는 법** — separator가 없으면 `제9조(근로시간)1주간의 소정근로시간은...`처럼 **태그 경계가 사라져서 단어가 붙어버립니다.**
문단 구분이 통째로 없어지면 나중에 청킹할 때 자를 지점을 못 찾습니다.
한 글자짜리 옵션이지만 파이프라인 끝까지 영향을 줍니다.

## 5. `db.py` — SQL을 읽어봅시다

PostgreSQL이 없어도 **SQL 자체는 읽을 수 있습니다.** 이 프로젝트에서 가장 중요한 설계 결정이 여기 있습니다.

In [ ]:
show("db.py", grep="CREATE TABLE|url |content_type|text_content|binary_content|crawled_at|ON CONFLICT|EXCLUDED")

**코드에서 짚을 곳** — 테이블 설계에서 눈여겨볼 것이 세 가지 있습니다.

**① `text_content`와 `binary_content`를 나눠둔 것** — HTML은 텍스트로, PDF는 원본 바이너리 그대로
저장합니다. PDF에서 글자를 뽑는 일은 **이 단계에서 하지 않습니다.** 다음 프로젝트(A-2)의 몫이죠.
왜냐하면 PDF 추출 도구를 나중에 바꾸고 싶어질 수 있고, 그때 원본이 남아 있어야 다시 돌릴 수 있으니까요.

**② `url`에 `UNIQUE`를 건 것** — 같은 페이지를 두 번 긁어도 행이 두 개 생기지 않습니다.

**③ `ON CONFLICT ... DO UPDATE` (UPSERT)** — 이미 있는 URL이면 새로 넣지 않고 **내용만 갱신**합니다.
규정은 개정되기 때문에, 재크롤링하면 최신 내용으로 덮어써야 합니다.

SQLite로 같은 구조를 만들어서 UPSERT가 실제로 어떻게 동작하는지 보겠습니다.
(PostgreSQL과 문법이 거의 같습니다. `SERIAL` → `INTEGER PRIMARY KEY AUTOINCREMENT`, `BYTEA` → `BLOB` 정도만 다릅니다.)

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    content_type TEXT NOT NULL,
    text_content TEXT,
    binary_content BLOB,
    crawled_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)
""")

UPSERT = """
INSERT INTO crawled_documents (url, content_type, text_content)
VALUES (?, ?, ?)
ON CONFLICT (url) DO UPDATE SET
    text_content = excluded.text_content,
    crawled_at = CURRENT_TIMESTAMP
"""

# 같은 URL을 두 번 저장해봅니다. 두 번째는 내용이 개정된 상황.
conn.execute(UPSERT, ("https://ex.com/rule", "html", "1주 40시간으로 한다."))
conn.execute(UPSERT, ("https://ex.com/rule", "html", "1주 36시간으로 한다. (개정)"))
conn.commit()

for row in conn.execute("SELECT id, url, text_content FROM crawled_documents"):
    print(row)
print("\n행 개수:", conn.execute("SELECT COUNT(*) FROM crawled_documents").fetchone()[0])

**결과 읽는 법**

**행이 하나만 남고 내용은 최신으로 바뀌었습니다.** UPSERT가 없었다면 재크롤링할 때마다
같은 문서가 쌓여서, 검색 결과에 옛날 버전과 새 버전이 같이 튀어나왔을 겁니다.
규정 챗봇에서 이건 치명적입니다. **폐지된 조항을 근거로 답하는 챗봇**이 되니까요.

## 6. 전체 흐름을 PostgreSQL 없이 돌려보기

이제 `crawl_one()`을 통째로 실행해봅니다. DB만 없으면 되니까,
**`save_document`를 우리 함수로 바꿔치기(몽키패칭)** 해서 SQLite에 넣게 만들겠습니다.

이건 노트북용 꼼수가 아니라 **테스트를 짤 때 실제로 쓰는 기법**입니다.
`crawl.py`가 `db.py`에 직접 SQL을 쓰지 않고 `save_document()` 함수 하나로 위임해둔 덕분에
이런 교체가 가능합니다. 파일을 나눠둔 보람이 여기서 나옵니다.

In [ ]:
import crawl

saved = []


def fake_save_document(url, content_type, text_content, binary_content):
    """db.save_document 대신 호출될 가짜 함수. 실제로는 PostgreSQL에 INSERT 합니다."""
    saved.append(
        {
            "url": url,
            "type": content_type,
            "text": (text_content or "")[:60],
            "binary_bytes": len(binary_content) if binary_content else 0,
        }
    )


crawl.save_document = fake_save_document  # 바꿔치기

# 진짜 네트워크 요청이 일어납니다. (example.com은 이런 테스트용으로 공개된 도메인입니다)
crawl.crawl_one("https://example.com")

for item in saved:
    print(item)

**결과 읽는 법**

`crawl_one()` 안의 코드는 **한 줄도 고치지 않고** 전체 경로가 돌았습니다.
`fetch()` → `extract_text_from_html()` → `save_document()`까지 진짜 프로젝트 코드입니다.

PDF 분기도 확인해봅시다. `crawl_one`은 URL이 `.pdf`로 끝나는지만 보고 갈래를 나눕니다.

In [ ]:
show("crawl.py", grep="endswith|save_document")

> ⚠️ **이 판별 방식의 한계도 같이 알아두세요.** URL이 `.pdf`로 끝나지 않는데 내용은 PDF인 경우
> (`/download?id=123` 같은 링크)를 놓칩니다. 실무에서는 응답의 `Content-Type` 헤더를
> 같이 보는 게 안전합니다. 연습 문제로 남겨두겠습니다.

## 정리

이 프로젝트를 한 문장으로 줄이면: **"웹에서 긁어온 원본을, 손대지 않고 그대로 DB에 쌓아두는 계층"** 입니다.

읽으면서 건진 설계 판단들:

| 결정 | 이유 |
|---|---|
| 원본을 그대로 보관 | 재처리할 때 다시 크롤링하지 않으려고 |
| PDF는 바이너리로 저장 | 추출 도구를 나중에 바꿀 수 있게 |
| `UNIQUE` + UPSERT | 재크롤링 시 중복 대신 갱신 (폐지 조항이 남지 않게) |
| `try/except`로 개별 실패 격리 | 1건 실패가 전체를 멈추지 않게 |
| `time.sleep()` | 상대 서버에 대한 예의 + 차단 방지 |
| DB 접근을 `db.py`로 분리 | 위에서 본 것처럼 교체·테스트가 쉬워짐 |

**남의 코드를 읽는 순서**도 같이 연습했습니다. 파일 목록 → 메인 함수 → 거기서 부르는 함수 순으로
내려가고, 중간에 "왜 이렇게 했지?" 싶은 곳에서 멈춰서 이유를 찾는 것.

**가장 기억할 것**: **원본은 손대지 말고 그대로 보관한다.**
정제와 청킹은 언제든 다시 할 수 있지만, 크롤링은 다시 못 할 수도 있습니다.

**스스로 확인해보기**

- [ ] 크롤링한 걸 바로 검색엔진에 넣지 않고 DB를 거치는 이유를 설명할 수 있다
- [ ] `<script>`를 안 지우면 검색 결과가 어떻게 망가지는지 말할 수 있다
- [ ] UPSERT가 없으면 규정이 개정됐을 때 무슨 일이 생기는지 안다
- [ ] `crawl_one()`을 `try/except`로 감싼 이유를 안다
- [ ] `save_document`를 몽키패칭해서 DB 없이 코드를 돌려봤다

## 연습 문제

**1. `Content-Type`으로 PDF를 판별하도록 고치기**
`crawl_one()`은 URL이 `.pdf`로 끝나는지만 봅니다. 응답 헤더의 `Content-Type`이
`application/pdf`인 경우도 PDF로 처리하도록 고쳐보세요.

**2. 크롤링 실패를 DB에 기록하기**
지금은 실패하면 화면에 출력만 하고 끝입니다. 어떤 URL이 언제 왜 실패했는지 남기려면
테이블과 코드를 어떻게 바꿔야 할까요? 스키마를 직접 설계해보세요.

**3. 변경 감지**
같은 URL을 다시 크롤링했을 때, 내용이 **실제로 바뀐 경우에만** `crawled_at`을 갱신하려면?
(힌트: 본문의 해시값을 컬럼으로 두면 비교가 쉬워집니다. 왜 이게 필요할까요?)

**해설/정답**: [01_crawl_storage_solutions.ipynb](01_crawl_storage_solutions.ipynb)

## 다음 단계

여기서 쌓은 원본을 꺼내 **정제 → 청킹 → 색인**하는 것이 다음 프로젝트입니다.
[`02_preprocess` 동행 노트북](../02_preprocess/02_preprocess.ipynb)으로 이어집니다.

라이브러리 자체를 더 파고들고 싶다면
[`rag-pipeline-practice/01_web_crawling`](../../rag-pipeline-practice/01_web_crawling/01_web_crawling.ipynb)에
`requests`/`BeautifulSoup`/`sqlite3` 실습이 따로 있습니다.